# The Paul-concordance extraction pipeline

Everything in this notebook documents and operates **one command**:

```bash
python3 paul_concordance.py --workers 30
```

That command is the whole extraction. It reads the shared JSTOR full-text corpus,
scans every article for references to Paul, and writes the concordance. This
notebook exists so that the command — and the two stages behind it — can be
understood, reproduced and tuned **without any agentic help**: every parameter is
in the CONFIG cell below, and every claim about the code is re-checked against the
code when you run it.

## The two stages

| stage | script | what it does |
| --- | --- | --- |
| 0 · corpus preparation | `/srv/data/jstor/jstor2026-04/jstor_fulltext_parser.py` | flattens the 6 GB gzip delivery into one `.txt` per item, once. Shared by every project using this JSTOR delivery. |
| 1 · extraction | `scripts/paul_concordance.py` | the concordance scan. Paul-specific, lives in this repo. |

Stage 0 is not part of this notebook's job — it is already done and shared. It is
described in §2 only because the concordance's `char_start`/`char_end` point into
its output.

## What this notebook does *not* do

It does not interpret the results. Descriptive tables of the output appear in §6,
but only as **verification** — to let you confirm the run did what it says. The
analysis is yours to do; §7 shows how to start reading the data on your own.

## 1 · CONFIG — everything you might want to change

Set these, then run §3. Every value here maps to a command-line flag of
`paul_concordance.py`, so the same run can be reproduced outside the notebook
(§8). The defaults reproduce the committed run.

In [ ]:
# ============================ CONFIG =====================================
# Paths.  The corpus is shared and read-only; all Paul-specific output is
# inside this project so it can never collide with another JSTOR project.
JSTOR        = Path("/srv/data/jstor/jstor2026-04")
FULLTEXTS    = JSTOR / "fulltexts_txts"                       # stage-0 output
METADATA_CSV = JSTOR / "jstor_metadata_2026-04-28-filtered.csv"
SCRIPT       = Path("/home/jupyter-vojta/notebooks/mops/scripts/paul_concordance.py")
SCAN_DIR     = Path("/home/jupyter-vojta/notebooks/mops/data/large_files/paul_scan")

# Which items to scan.  Leave a filter empty/0 to switch it off.
TIERS       = "ABC"     # any subset of A, B, C            --tier
JOURNAL     = ""        # substring of is_part_of          --journal
LANGUAGE    = ""        # language code prefix, e.g. "eng" --language
YEAR_FROM   = 0         #                                  --year-from
YEAR_TO     = 9999      #                                  --year-to
LIMIT       = 0         # 0 = all 384,807 documents        --limit

# Detector tunables.
CTX         = 46        # flag-detection window, chars      --ctx
KWIC        = 42        # KWIC window, chars                --kwic
APPARATUS   = 3         # citation-hint agreement for the
                        #   citation_apparatus flag         --apparatus-threshold
TIER_C_CAPITALS = "keep"  # what to do with tier-C hits whose
                        #   follow-word is Capitalised:
                        #   "keep" (flag them) | "drop"    --tier-c-capitals

# Runtime.
WORKERS     = 30        # processes                        --workers
TAG         = ""        # suffix for output filenames      --tag
# =========================================================================

OUTDIR = SCAN_DIR

### 1.1 The command, built from CONFIG

No hidden steps: this cell assembles the exact argv that §3 executes, so you can
copy it and run it yourself, unchanged, from a shell.

In [ ]:
def build_command() -> list[str]:
    """Turn CONFIG into the argv of paul_concordance.py."""
    cmd = [sys.executable, str(SCRIPT), "--workers", str(WORKERS)]
    if TIERS != "ABC":
        cmd += ["--tier", TIERS]
    if JOURNAL:
        cmd += ["--journal", JOURNAL]
    if LANGUAGE:
        cmd += ["--language", LANGUAGE]
    if YEAR_FROM:
        cmd += ["--year-from", str(YEAR_FROM)]
    if YEAR_TO != 9999:
        cmd += ["--year-to", str(YEAR_TO)]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    if CTX != 46:
        cmd += ["--ctx", str(CTX)]
    if KWIC != 42:
        cmd += ["--kwic", str(KWIC)]
    if APPARATUS != 3:
        cmd += ["--apparatus-threshold", str(APPARATUS)]
    if TIER_C_CAPITALS != "keep":
        cmd += ["--tier-c-capitals", TIER_C_CAPITALS]
    if TAG:
        cmd += ["--tag", TAG]
    return cmd


COMMAND = build_command()
print(" ".join(COMMAND))

## 2 · Stage 0: how the corpus the scan reads was produced

Stage 0 ran once, on 2026-09-04, and its output is shared by every project using
this JSTOR delivery. It is: **one `.txt` per item**, in a flat directory.

```
/srv/data/jstor/jstor2026-04/fulltexts_txts/<item_id>.txt     all OCR pages, unpaged
/srv/data/jstor/jstor2026-04/references_txts/<item_id>.txt    one bibliography entry per line
```

### Why it exists

The delivery is a **single 6.0 GB gzip stream** of 384,807 JSON Lines:

```json
{"iid": "...", "full_text": ["<page 1>", "<page 2>", "..."], "references": ["<ref>", "..."]}
```

Gzip blocks are not independently seekable and JSTOR ships no offset index, so
extracting one article from the archive costs a **full sequential decompression —
measured ~105 s**. Flattening to files pays that cost once; afterwards opening any
article is a ~0.04 ms file read. The scan needs to read all 384,807 documents, so
without stage 0 it would be unusable.

### The documented facts about this archive

Verified over all 384,807 records:

* three keys only, and `iid` is always the first — which is why a raw-byte slice
  `line[8:44]` yields the id and lets the extractor reject unwanted lines
  *before* paying for `json.loads`;
* `full_text` is always a list of strings, never empty, with no page-marker tokens;
* `references` holds 14,186,773 entries across the delivery, and **105,322 records
  (27.4%) have an empty list** — those get no file, so `references_txts/` is
  legitimately smaller than `fulltexts_txts/`;
* `iid` == `item_id` in the metadata CSV; 22 CSV rows have no text at all.

### Normalisation policy — what stage 0 deliberately does *not* do

The archive is OCR output that the research depends on being faithful to, so the
default is conservative. Measured on the first 20,000 records:

| phenomenon | frequency | handling |
| --- | --- | --- |
| HTML entities (`&amp;`) | 35 hits in 8 docs (0.04%) | unescaped by default; `--raw` keeps them |
| entities inside references | 1,140 of 14,625 docs with refs (7.8%) | unescaped by default |
| inline tags `<sup> <i> <em> <br>` | 2,065 in 488 docs (2.4%) | **kept** — stripping risks mangling genuine `<` in philological text; `--strip-tags` is opt-in |
| U+FFFD replacement char | 259 in 14 docs | kept (already lossy in the source) |

The only unconditional normalisation is CRLF→LF, dropping whitespace-only pages,
trimming, and collapsing runs of 3+ newlines. Everything else is opt-in, so the
corpus stays reproducible from the archive.

### Verified result of the stage-0 run

```json

In [ ]:
# Stage 0's own report, read from disk rather than restated.
extract_report = json.loads((JSTOR / "extract_logs" / "extract_report.json").read_text())
for k in ("elapsed_seconds", "docs_per_second", "lines_seen", "records_parsed",
          "text_files_written", "reference_files_written", "records_without_references",
          "chars_text_total", "longest_document"):
    print(f"  {k:28s} {extract_report.get(k)}")
print(f"  {'errors':28s} {extract_report['errors']}")

> **Caveat recorded in the report itself:** the `extract_report.json` on disk was
> restored from `extract_logs/extract.log`, because a later `--limit 3000` resume
> run overwrote the report with its own statistics. The numbers above are the
> original full run's; the `note` field in the JSON says so.

The one number that matters for this notebook is the empty-file count: **295 files
are empty** because their OCR payload was whitespace only. They are still files,
so the scan opens them and finds nothing — no special case needed.

## 3 · Stage 1: the extraction, step by step

Run the cell below, or reproduce it in a shell with the printed command. It takes
about **5 minutes** for the full corpus.

Then read §3.1–§3.6, which explain what just happened.

In [ ]:
import subprocess, time

if OUTDIR != SCAN_DIR:
    OUTDIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
proc = subprocess.run(build_command(), capture_output=True, text=True)
print(proc.stdout[-4000:])
if proc.stderr:
    print("--- stderr ---")
    print(proc.stderr[-2000:])
print(f"\nexit code {proc.returncode}   wall clock {time.time() - t0:.0f}s")

### 3.1 Where the time goes

A single sequential pass over a 6 GB gzip. Roughly:

| phase | cost |
| --- | --- |
| gzip decompression | ~150 s (single-threaded, the floor — nothing to parallelise) |
| regex matching, 14 patterns × 384,807 docs | ~150 s across 30 worker processes |
| writing `hits.tsv.gz` + 3 concordances + per-doc table | seconds |

Because decompression is the floor, adding workers beyond ~30 buys little. The
process pool uses `fork`, so the tunables set in `main()` are inherited by the
children.

### 3.2 The matching pipeline, per document

Each worker does the same six steps. They are worth knowing because three of them
are places where a subtle bug can hide.

**1 — cheap prefilter.** `if "aul" not in text: skip`. 73% of documents contain no
`aul` substring at all, so two thirds of the corpus never reaches the regexes.

**2 — two derived copies, both length-preserving.**

| copy | transformation | why |
| --- | --- | --- |
| `flat` | HTML tags and entities replaced by **equal-width runs of spaces**; whitespace, quotes and dashes normalised to single characters | removes markup without shifting offsets |
| `norm` | `flat` lowercased | case-insensitive matching without `re.I` surprises |

Both keep byte-for-byte length equality with the file on disk, which is what makes
`char_start`/`char_end` in the output valid offsets into
`fulltexts_txts/<item_id>.txt`. A quote taken from those offsets is *verbatim*.

> **This lowercasing is the single most dangerous property of the design.** Any
> test that depends on letter case cannot be expressed as a regex here, because by
> the time a regex runs, `Paul Lorenzen` and `Paul wrote` are identical. §4 is
> entirely about a bug caused by exactly this.

**3 — all 14 patterns run, then overlaps are resolved.** A hit that lies inside a
strictly better tier's span is dropped. Spans are sorted best-tier-first,
longest-first, so the first span to claim a mention is the most specific
description of it. This is why `Pauline letters` counts once as `A5`, not twice as
`A5` + `B1`.

**4 — tier demotion.** A hit whose own pattern is tier A or B, but which is
followed by a name-like Capital, and whose neighbourhood contains an excluded
surname, is **demoted** one tier (A→B, B→C) and flagged `demoted_name_after`. It is
never deleted: 4,507 rows take this path. So `tier:` totals in the report differ
slightly from `pattern:` totals by construction.

**5 — the case-sensitive test that regexes cannot do.** For `C1` only, the first
letter after the match is inspected **in the original text** and must be lowercase.
This is §4.

**6 — flags.** Never filters. Every hit keeps its row; the flags say *why to be
careful*. A flag is a warning label for you, not a decision made for you — except
where a switch explicitly says otherwise (§1 CONFIG, and `--tier-c-capitals`).

### 3.3 The 14 patterns and the three tiers

`A` = explicit identification, `B` = strong, `C` = bare *Paul* resolved by context.
Read them as a confidence ladder, **not** three parallel counts: a hit appears once,
in the best tier that fired.

| pattern | tier | matches |
| --- | --- | --- |
| `A1_apostle_pre` | A | `Apostle Paul`, up to two intervening words |
| `A2_apostle_post` | A | `Paul the apostle` |
| `A3_apostle_foreign` | A | `apôtre/apostel/apóstol/apostolo … Paul`, both orders |
| `A4_tarsus` | A | `Paul of/van/von/de Tarsus`, `Tarsian` |
| `A5_pauline_writings` | A | *Pauline* + writing noun, `letters of Paul`, `epistula Pauli` |
| `A6_saint_paul` | A | `St./Saint/Sanctus/Sankt/San … Paul` |
| `B1_pauline_adj` | B | bare adjectival *Pauline* and its inflections |
| `B2_paulus` | B | Latin *Paulus* |
| `B3_of_paul` | B | genitive `of/de/von/van … Paul` |
| `B4_paul_genitive_abstract` | B | `Paul's` + a theological noun (theology, christology, …) |
| `C1_paul_then_lowercase` | C | bare *Paul* + a **lowercase** word — see §4 |
| `C2_paul_possessive` | C | bare `Paul's` |
| `C3_paul_verb` | C | bare *Paul* + a speech/action verb |
| `C4_the_paul` | C | preposition or determiner + bare *Paul* |

Two deliberate choices in the regex vocabulary, both from measurement:

* The adjectival stem is a **whitelist**, not `paulin\w*`. The wildcard also
  catches `Paulinus` (a Church father), `Paulina`/`Paulino` (given names, towns),
  `Pauling` (surname) and `Paulinum` (place) — 2,459 + 1,248 + 1,673 + 678 hits in
  the corpus, all wrong.
* `St./Saint Paul` is **included in tier A by decision**, because in this corpus it
  is overwhelmingly the apostle. Building and city senses are *flagged*
  (`place_or_building`, 3,890 rows), never dropped.

## 4 · The C1 bug — why tier C shrank by 145,000 hits

This is the most important section in the notebook. It documents a real defect,
because anyone rebuilding the concordance needs to know it is fixed and how to
check that it stays fixed.

### 4.1 What was wrong

`C1_paul_then_lowercase` is defined as "bare *Paul* followed by a lowercase word",
and its original regex expressed the case test as a lookahead:

```python
# ORIGINAL — broken
("C1_paul_then_lowercase", "C", rf"\b{PAUL}\b{G}(?=[a-zà-ÿ])"),
```

Two independent defects made that lookahead meaningless:

| # | defect | consequence |
| --- | --- | --- |
| 1 | every pattern is compiled with **`re.IGNORECASE`**, and under `re.I` the class `[a-zà-ÿ]` also matches capitals | `[a-zà-ÿ]` was never a lowercase test |
| 2 | patterns are matched against `norm`, which is **`flat.lower()`** | by then the entire document is lowercase, so the test is vacuous even without `re.I` |

Defect 2 is the fatal one: the test **could not work as written under any flag**,
because the information it needed had already been destroyed. Measured on real
output, the pattern was capturing bibliography entries and other people named Paul:

```
match='Paul '   kwic_left='… Oxford, 1971. [2]'          kwic_right='Lorenzen, Differential…'
match='Paul, '  kwic_left='… Routledge & Kegan'          kwic_right='London and New York, 1961…'
match='Paul '   kwic_left='… A Critique of Jean-'        kwic_right="Sartre's Ontology 35 (1951)…"
match='Paul '   kwic_left='… Present also were Bishops'   kwic_right='Reeves of Georgia and…'
```

### 4.2 The fix

Three changes, in order of importance:

1. **The case test moved out of the regex** into `_followed_by_lowercase()`, which
   runs on the **original** text (`C1_NEEDS_LOWERCASE_FOLLOW` in the script).
   A regex cannot express this, so it is no longer attempted.
2. **`C1` now requires a letter to follow at all** — `(?=[^\W\d_])` — so punctuation
   alone cannot satisfy it.
3. A hit failing the test is **dropped and counted** under
   `dropped:not_lowercase_follow` in the run report, so the size of the removal is
   auditable rather than invisible.

### 4.3 Effect, measured

Same corpus, same command, before and after:

| | before | after | change |
| --- | --- | --- | --- |
| tier A | 76,163 | 76,163 | **0** |
| tier B | 145,240 | 145,240 | **0** |
| tier C | 350,493 | 205,514 | −144,979 |
| `C1` hits | 250,661 | 105,682 | −144,979 |
| total hits | 571,896 | 426,917 | −144,979 |
| articles with ≥1 hit | 106,005 | 69,709 | −36,296 |
| `dropped:not_lowercase_follow` | — | 145,166 | (new counter) |

Tiers A and B are **bit-identical**, which is the expected result: the change
touches only the `C1` follow-word test. (145,166 drops vs a −144,979 net change:
the 187 difference is documents or hits that re-entered tier C from the demotion
path once their better-tier neighbours changed.)

### 4.4 Guard against regression

If anyone ever "simplifies" `C1` back into a regex lookahead, the check below
fails. Run it after any change to the pattern list.

In [ ]:
# ---- regression guard for the C1 case test -------------------------------
import importlib.util

spec = importlib.util.spec_from_file_location("paul_concordance", SCRIPT)
pc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pc)

# (a) the helper itself must separate the two cases on ORIGINAL text
cases = [("Paul wrote to the Romans", True),  ("Paul Lorenzen, Differ", False),
         ("Paul, A. & Shenhav",        False), ("Paul à ce sujet", True),
         ("Paul Sartre argues",        False), ("Paul Ricoeur", False)]
print("_followed_by_lowercase on original text:")
ok = True
for text, expected in cases:
    i = text.index("Paul") + 4
    got = pc._followed_by_lowercase(text, i)
    flag = "ok " if got == expected else "BAD"
    ok &= got == expected
    print(f"  {flag} {text!r:28s} -> {got}   (expected {expected})")

# (b) the pattern that stands in for C1 must not be case-sensitive by itself
c1 = dict((n, rx) for n, t, rx in pc.COMPILED)["C1_paul_then_lowercase"]
print(f"\nC1 regex is now: {c1.pattern}")
print("  it deliberately has NO case lookahead -- the test is in _followed_by_lowercase")
print(f"\n(a) passed: {bool(ok)}")

In [ ]:
# ---- and the output must contain no capital-after C1 rows ----------------
CAP_AFTER = re.compile(r"^[,\s;:.\u2026]*[^\W\d_]", re.UNICODE)

def c1_capital_rows(path: Path) -> tuple[int, int]:
    """(#C1 rows, #of those whose next word is Capitalised) in a hits table."""
    n = cap = 0
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for row in csv.DictReader(fh, delimiter="\t"):
            if not row["pattern"].startswith("C1"):
                continue
            n += 1
            rest = row["kwic_right"].lstrip("\u2026 ")
            m = CAP_AFTER.match(rest)
            if m and m.group(0)[-1:].isupper():
                cap += 1
    return n, cap

if (OUTDIR / "hits.tsv.gz").exists():
    n, cap = c1_capital_rows(OUTDIR / "hits.tsv.gz")
    print(f"C1 rows                 {n:>8,}")
    print(f"  followed by a Capital {cap:>8,}")
    print(f"\n{'PASS' if cap == 0 else 'FAIL'} — capital-after C1 rows must be 0")
else:
    print("no hits table yet — run §3 first")

## 5 · What the command writes

Into `paul_scan/` (this project, `data/large_files/paul_scan/`):

| file | contents |
| --- | --- |
| `hits.tsv.gz` | **one row per detected reference** — the primary output |
| `hits_by_doc.tsv.gz` | one row per article: `tier_a`, `tier_b`, `tier_c`, `total` |
| `concordance_A.txt` / `_B.txt` / `_C.txt` | aligned KWIC listing, one hit per line, meant to be read or grepped |
| `scan_report.json` | totals per tier, pattern and flag — the audit trail |
| `paul_exclusions.txt` | **input**, not output: the excluded-surname list, hand-maintained |
| `probes/` | the measurement scripts and outputs behind the pattern design |
| `paul_eval.tsv` | stratified sample for precision labelling (`verdict` column empty) |

### 5.1 Column reference for `hits.tsv.gz`

15 columns, one row per hit. These are exactly `paul_concordance.FIELDS`.

| column | meaning |
| --- | --- |
| `item_id` | joins to `fulltexts_txts/<item_id>.txt` and to the metadata CSV |
| `year`, `bidecade` | from `published_date`; `bidecade` is a 20-year start year, e.g. `1980-1999` |
| `journal` | `is_part_of` from the metadata |
| `language` | `languages` from the metadata, e.g. `eng` |
| `tier` | `A` / `B` / `C` |
| `pattern` | which of the 14 patterns fired |
| `flags` | `;`-separated warnings (§5.2) — empty string when clean |
| `char_start`, `char_end` | **verbatim offsets into the file on disk** |
| `match` | the matched text as it appears in the original |
| `kwic_left`, `kwic_right` | ±42 chars of context, cut at word boundaries, `…` when truncated |
| `sentence` | abbreviation-aware sentence containing the hit, ≤400 chars |
| `doc_len` | length of the whole document in characters |

### 5.2 The flags

Flags never remove a row. They tell you why a row deserves a second look.

| flag | meaning | typical tier |
| --- | --- | --- |
| `citation_apparatus` | ≥3 citation-style regexes agree nearby — usually a bibliography | C |
| `excluded_name` | a surname from `paul_exclusions.txt` sits within ±46 chars | C |
| `demoted_name_after` | *also* name-like; the row was demoted one tier | B, C |
| `capital_after` | followed by a capitalised word (the `Paul Sartre` shape) | C |
| `papacy_or_numeral` | `Pope … Paul`, or `Paul II/VI` | C |
| `place_or_building` | `St Paul's Cathedral`, `Paul Street`, `Kegan Paul`-like | A, B |
| `possibly_given_name` | adjectival *Pauline* before a name (`Pauline Glaubinger`) | B |
| `initial_before` / `initial_after` | an initial abuts the name (`P. Paul`, `Paul M.`) | C |
| `running_head` | the line is >85% uppercase — a page header caught in the OCR | any |

In [ ]:
# The output tree, and the size of each part.
for p in sorted(OUTDIR.rglob("*")):
    if p.is_file() and "scratch" not in p.parts:
        rel = p.relative_to(OUTDIR)
        print(f"  {str(rel):34s} {p.stat().st_size / 1e6:>9.2f} MB")

## 6 · Verifying the run

These cells re-derive the run's headline numbers **from the output** and compare
them with `scan_report.json`. This is a check that the extraction did what it
claims — not an analysis of Paul.

If a number here disagrees with the report, the run is not trustworthy; that is
the whole purpose of the section.

In [ ]:
import gzip, csv, json
from collections import Counter

hits    = pd.read_csv(OUTDIR / "hits.tsv.gz", sep="\t", low_memory=False)
by_doc  = pd.read_csv(OUTDIR / "hits_by_doc.tsv.gz", sep="\t", low_memory=False)
report  = json.loads((OUTDIR / "scan_report.json").read_text())
print(f"report generated {report['generated']}  in {report['seconds']}s with "
      f"{report['tiers']} tiers")

In [ ]:
# --- tier and pattern totals: recomputed vs reported ----------------------
recomputed_tier = hits["tier"].value_counts().sort_index()
reported_tier = {k.split(":")[1]: v for k, v in report["totals"].items() if k.startswith("tier:")}

check = pd.DataFrame({
    "recomputed": recomputed_tier,
    "reported": pd.Series(reported_tier),
})
check["match"] = check["recomputed"] == check["reported"]
print("tiers")
print(check.to_string())

pat_recomputed = hits["pattern"].value_counts()
pat_reported = pd.Series({k.split(":", 1)[1]: v for k, v in report["totals"].items()
                          if k.startswith("pattern:")})
pcheck = pd.DataFrame({"recomputed": pat_recomputed, "reported": pat_reported})
pcheck["match"] = pcheck["recomputed"] == pcheck["reported"]

# pattern totals legitimately differ from tier totals because of demotion (§3.2)
pat_total = int(pat_recomputed.sum())
n = len(hits)
print(f"\npattern rows sum to {pat_total:,} vs {n:,} hit rows — difference {pat_total - n:,}")
print("  (hits demoted out of their pattern's tier; §3.2 step 4)")
print(f"\nall tier totals reconciled: {bool(check['match'].all())}")
print(f"all pattern totals reconciled: {bool(pcheck['match'].all())}")

In [ ]:
# --- document-level reconciliation ---------------------------------------
docs_scanned = report["documents_scanned"]
print(f"documents scanned            {docs_scanned:>9,}")
print(f"documents with >=1 hit       {len(by_doc):>9,}   {len(by_doc)/docs_scanned:>8.1%}")
print(f"  reported                   {report['documents_with_hits']:>9,}   "
      f"{'match' if report['documents_with_hits'] == len(by_doc) else 'MISMATCH'}")

print(f"\nper-tier document counts")
for tier in "ABC":
    got = hits.loc[hits["tier"] == tier, "item_id"].nunique()
    want = report["docs_with_hits_per_tier"][tier]
    print(f"  tier {tier}   recomputed {got:>8,}   reported {want:>8,}   "
          f"{'match' if got == want else 'MISMATCH'}")

print(f"\nby_doc totals reconcile with hits: "
      f"{bool(int(by_doc['total'].sum()) == len(hits))}")

In [ ]:
# --- the drops the fix introduced, straight from the report ---------------
dropped = {k.split(":", 1)[1]: v for k, v in report["totals"].items()
           if k.startswith("dropped:")}
if dropped:
    print("hits removed during the scan (counted, not silent):")
    for k, v in sorted(dropped.items(), key=lambda kv: -kv[1]):
        print(f"  {k:32s} {v:>9,}")
else:
    print("no dropped:* counters in this report")

print(f"\nflags still attached to surviving rows:")
flag_rows = {k.split(":", 1)[1]: v for k, v in report["totals"].items()
             if k.startswith("flag:")}
print(pd.Series(flag_rows).sort_values(ascending=False).to_string())

## 7 · Exploring the concordance yourself

The two tables are ordinary TSVs; `pandas`, `duckdb`, `grep` and your eyes all
work. Below are the tools the pipeline ships for reading, and a couple of sane
starting points. Everything returns rows you can filter, so nothing here commits
you to a definition of "a reference to Paul".

### Reading the concordances as text

`concordance_A.txt` is the tier-A concordance in the printed-concordance shape —
one hit per line, context aligned around the match. It is meant to be **read**:

```bash
grep -n "of Tarsus" data/large_files/paul_scan/concordance_A.txt | head
less data/large_files/paul_scan/concordance_A.txt
```

In [ ]:
# The built-in readers.  `paul_hits()` filters; `paul_concordance()` prints an
# aligned KWIC listing of whatever it returns.
_hits_cache = None

def paul_hits(tier=None, pattern=None, journal=None, language=None,
              year_from=None, year_to=None, exclude_flags=("citation_apparatus",),
              drop_flags=("excluded_name",)):
    """Filter the hits table.  Loads once (~15 s), then cached."""
    global _hits_cache
    if _hits_cache is None:
        _hits_cache = pd.read_csv(OUTDIR / "hits.tsv.gz", sep="\t", low_memory=False)
    h = _hits_cache
    if tier:
        h = h[h["tier"].isin(list(tier.upper()))]
    if pattern:
        pats = (pattern,) if isinstance(pattern, str) else tuple(pattern)
        h = h[h["pattern"].str.startswith(pats)]
    if journal:
        h = h[h["journal"].str.contains(journal, case=False, na=False)]
    if language:
        h = h[h["language"].fillna("").str.startswith(language)]
    if year_from:
        h = h[pd.to_numeric(h["year"], errors="coerce") >= year_from]
    if year_to:
        h = h[pd.to_numeric(h["year"], errors="coerce") <= year_to]
    flags = h["flags"].fillna("")
    for f in drop_flags or ():
        h = h[~flags.reindex(h.index).str.contains(f, regex=False)]
    for f in exclude_flags or ():
        h = h[~flags.reindex(h.index).str.contains(f, regex=False)]
    return h


def paul_concordance(frame, width=44):
    """Print an aligned KWIC listing, as in the concordance files."""
    for r in frame.itertuples():
        print(f"{str(r.kwic_left)[-width:]:>{width}} \u27ea{r.match}\u27eb "
              f"{str(r.kwic_right)[:width]:<{width}}  {r.year} {str(r.journal)[:26]}")


def paul_context(row, window=400):
    """The surrounding document text, read via the offsets in the hit row."""
    text = (FULLTEXTS / f"{row['item_id']}.txt").read_text(encoding="utf-8")
    a = int(row["char_start"])
    return text[max(0, a - window):a + window]

In [ ]:
# Have a look.  Nothing here is a finding -- it is a sanity read.
paul_concordance(paul_hits(tier="A").sample(12, random_state=0))

In [ ]:
# Per-pattern picture, with the flags that matter for the strict claims.
summary = (hits.assign(cit=hits["flags"].fillna("").str.contains("citation_apparatus"),
                       exc=hits["flags"].fillna("").str.contains("excluded_name"))
               .groupby(["tier", "pattern"])
               .agg(hits=("item_id", "size"),
                    docs=("item_id", "nunique"),
                    in_apparatus=("cit", "sum"),
                    name_flag=("exc", "sum"))
               .sort_values("hits", ascending=False))
summary["clean"] = summary["hits"] - summary["in_apparatus"] - summary["name_flag"]
print(summary.to_string())

In [ ]:
# How many articles at each confidence level, and the strict interpretation of
# each tier as a headline number.  Choose your own threshold; this just shows
# that all three are derivable from one table.
print("articles, by the best tier they reach:")
for label, tiers in [("A only (explicit)",        ["A"]),
                     ("A or B (explicit+strong)", ["A", "B"]),
                     ("A, B or C (any bare Paul)", ["A", "B", "C"])]:
    print(f"  {label:28s} {hits[hits['tier'].isin(tiers)]['item_id'].nunique():>8,}")

print("\nthe same, after dropping the rows flagged as bibliography/name:")
strict = hits[~hits["flags"].fillna("").str.contains("citation_apparatus|excluded_name")]
for label, tiers in [("A only (explicit)",        ["A"]),
                     ("A or B (explicit+strong)", ["A", "B"]),
                     ("A, B or C (any bare Paul)", ["A", "B", "C"])]:
    print(f"  {label:28s} {strict[strict['tier'].isin(tiers)]['item_id'].nunique():>8,}")

In [ ]:
# One document, read end to end from a hit -- the verbatim-offset guarantee.
row = hits[hits["pattern"].str.startswith("A4")].iloc[0]
print(f"{row['item_id']}  {row['year']}  {row['journal']}")
print(f"offsets {row['char_start']}..{row['char_end']}   match {row['match']!r}\n")
print(paul_context(row))

## 8 · Reproducing outside the notebook

The notebook is a convenience wrapper. The run is reproducible from a shell with
no Python-notebook machinery at all:

```bash
cd /home/jupyter-vojta/notebooks/mops/scripts

# the full run, exactly as committed
python3 paul_concordance.py --workers 30

# a 2-minute smoke test on the first 40,000 documents, writing *_probe.tsv.gz
python3 paul_concordance.py --limit 40000 --workers 30 --tag probe

# only tier A, only English, only Novum Testamentum, 1860-1920
python3 paul_concordance.py --tier A --language eng --journal "Novum Testamentum" \
        --year-from 1860 --year-to 1920

# see every tunable
python3 paul_concordance.py --help
```

Check a completed run:

```bash
python3 paul_concordance.py --verify        # file counts and sizes
cat data/large_files/paul_scan/scan_report.json
```

### Inputs you must keep in the project

| file | role |
| --- | --- |
| `scripts/paul_concordance.py` | the detector |
| `data/large_files/paul_scan/paul_exclusions.txt` | excluded surnames — **edit this to change behaviour, not the code** |
| `data/large_files/paul_scan/paul_eval.tsv` | labelling sample for precision work |

`srv/data/jstor/...` is shared, read-only input: the full texts, the metadata CSV,
and `jstor_fulltext_parser.py`.

### Deliberate limitations — know these before quoting a number

* **Greek is not matched.** `Παῦλος` / `Σαῦλος` need polytonic normalisation that
  `normalise()` does not do.
* **`references_txts/` is not scanned.** Only the article body. A hit whose flag
  is `citation_apparatus` is therefore usually an author name inside a *bibliography
  that sits in the body text*, not an entry from `references_txts/`.
* **Tier C is context-resolved, not verified.** It is the tier to treat with most
  caution, and `paul_eval.tsv` exists to measure its precision.
* **OCR damage defeats the regexes in places.** `Paul s` for `Paul's` is handled;
  `PauP` is not.
* **The place-word list is English-biased.** `Cathédrale S. Paul de Londres` is not
  flagged `place_or_building`.
* **`St. Paul` is included in tier A by decision.** If you disagree, restrict to
  `--tier A` minus `A6`, or filter the pattern out in pandas.

In [ ]:
# ---- environment recap, for the record ----------------------------------
print("python      ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("script      ", SCRIPT)
print()
import hashlib
for p in (SCRIPT, OUTDIR / "paul_exclusions.txt"):
    h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    print(f"  {h}  {p.relative_to(p.parents[2]) if len(p.parents) > 2 else p}")